# RepoCoder Studio – Stage 2 Notebook
## Natural Language → Python Code Generation

Final Stage 2 design aligned with Stage 1 learnings:

- Stable dependency setup
- Dataset loading with fallbacks where needed
- Dataset normalization
- Dataset cleaning and task-alignment filtering
- Token-length analysis before training
- Prompt/code-only supervision
- Baseline evaluation before fine-tuning
- Qwen pretrained baseline
- DeepSeek pretrained baseline
- Qwen LoRA fine-tuned model
- Training on MBPP + CodeAlpaca
- Evaluation on MBPP Test + HumanEval
- Pass@k, Execution Accuracy, and optional CodeBLEU
- Saved experiment artifacts for report writing

In [1]:
# ============================================================
# CODE BLOCK 1: Stable environment setup
# ============================================================

!pip uninstall -y numpy pandas scipy scikit-learn
!pip install -q --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  scipy==1.11.4 \
  scikit-learn==1.4.2

!pip install -q --no-cache-dir \
  torch \
  transformers==4.44.2 \
  datasets==2.21.0 \
  accelerate==0.34.2 \
  peft==0.12.0 \
  evaluate==0.4.2 \
  huggingface_hub==0.25.2 \
  tqdm \
  rouge-score \
  sentencepiece \
  protobuf

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 101.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 199.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 221.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 209.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 210.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into accou

In [48]:
# ============================================================
# CODE BLOCK 2: Imports and practical Colab configuration
# ============================================================

# -----------------------------
# Standard library imports
# -----------------------------
import os          # File and directory operations
import re          # Regular expressions for text cleaning/pattern matching
import gc          # Garbage collection (manual memory cleanup)
import ast         # Abstract Syntax Trees (Python syntax validation)
import json        # JSON serialization/deserialization
import math        # Mathematical operations
import random      # Random sampling (with reproducibility)
import tempfile    # Temporary file handling
import subprocess  # Running shell commands
import textwrap    # Formatting long strings neatly
from datetime import datetime  # Timestamps for logs/results
from pathlib import Path       # Cleaner path handling

# -----------------------------
# Third-party libraries
# -----------------------------
import numpy as np              # Numerical operations
import pandas as pd             # Tabular data handling
from tqdm.auto import tqdm      # Progress bars for loops

import torch                    # PyTorch deep learning framework
import transformers             # Hugging Face Transformers library
import datasets                 # Hugging Face Datasets library
import accelerate               # Hugging Face Accelerate (multi-GPU/TPU training)
import huggingface_hub          # Hugging Face Hub integration

# -----------------------------
# Hugging Face dataset/model utilities
# -----------------------------
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,                   # Tokenizer loader
    AutoModelForCausalLM,            # Model loader for causal language modeling
    TrainingArguments,               # Training configuration
    Trainer,                         # High-level training loop
    DataCollatorForLanguageModeling, # Handles batching and masking
    set_seed                         # Ensures reproducibility
)

# -----------------------------
# Parameter-efficient fine-tuning (PEFT)
# -----------------------------
from peft import (
    LoraConfig,      # LoRA configuration
    get_peft_model,  # Wraps base model with LoRA adapters
    TaskType         # Defines task type (e.g., causal LM)
)

# -----------------------------
# Environment checks
# -----------------------------
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("cuda:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# -----------------------------
# Reproducibility setup
# -----------------------------
SEED = 42
set_seed(SEED)          # Hugging Face seed
random.seed(SEED)       # Python random seed
np.random.seed(SEED)    # NumPy random seed

# ------------------------------------------------------------
# Practical run setup
# ------------------------------------------------------------

DEMO_MODE = True

RUN_QWEN_BASELINE = True
RUN_DEEPSEEK_BASELINE = False
RUN_FINE_TUNING = True

QWEN_MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
DEEPSEEK_MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

MBPP_DATASET_NAME = "mbpp"
HUMANEVAL_DATASET_NAME = "openai_humaneval"
EVALPLUS_MBPP_NAME = "evalplus/mbppplus"

MBPP_TRAIN_EXAMPLES = None
VALIDATION_EXAMPLES = None

TEST_EXAMPLES = 30
NUM_GENERATIONS = 3
PASS_K_VALUES = [1, 3]

MAX_PROMPT_LENGTH = 512
MAX_TOTAL_LENGTH = 768
MAX_NEW_TOKENS = 256

# Safer fine-tuning settings to reduce catastrophic forgetting
FT_LEARNING_RATE = 2e-5
FT_NUM_EPOCHS = 1
FT_WARMUP_RATIO = 0.10
FT_WEIGHT_DECAY = 0.01
FT_LORA_R = 8
FT_LORA_ALPHA = 16
FT_LORA_DROPOUT = 0.10

MIN_INSTRUCTION_WORDS = 3
MAX_INSTRUCTION_WORDS = 200
MIN_CODE_CHARS = 10
MAX_CODE_CHARS = 4000

OUTPUT_DIR = "/content/RepoCoderStudio_Stage2_MBPP_SafeFT"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "evaluation_results")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


torch: 2.11.0+cu128
transformers: 4.44.2
datasets: 2.21.0
accelerate: 0.34.2
huggingface_hub: 0.25.2
cuda: True
gpu: Tesla T4


In [49]:
# ============================================================
# CODE BLOCK 3: Dataset loading helpers
# ============================================================

def load_dataset_safe(dataset_name, config_name=None):
    try:
        if config_name is None:
            print(f"Loading dataset: {dataset_name}")
            return load_dataset(dataset_name)
        else:
            print(f"Loading dataset: {dataset_name}, config={config_name}")
            return load_dataset(dataset_name, config_name)

    except Exception as error:
        print(f"Failed to load dataset: {dataset_name}")
        print("Reason:", error)
        return None


def get_split_or_first(dataset_dict, preferred="train"):
    if dataset_dict is None:
        return None

    if preferred in dataset_dict:
        return dataset_dict[preferred]

    return dataset_dict[list(dataset_dict.keys())[0]]


def limit_dataset(ds, n, name="dataset"):
    if ds is None:
        return None

    if n is None:
        print(f"{name}: using full dataset ({len(ds)} examples)")
        return ds

    n = min(n, len(ds))
    print(f"{name}: using {n}/{len(ds)} examples")
    return ds.select(range(n))

In [50]:
# ============================================================
# CODE BLOCK 4: Load MBPP, HumanEval, and EvalPlus MBPP+
# ============================================================

# ------------------------------------------------------------
# Load datasets safely using a helper function (load_dataset_safe).
# This wrapper should catch errors (e.g., missing dataset, network issues)
# and return None instead of crashing. That way we can handle failures gracefully.
# ------------------------------------------------------------
mbpp_raw = load_dataset_safe(MBPP_DATASET_NAME)              # MBPP: Mostly Basic Python Problems
humaneval_raw = load_dataset_safe(HUMANEVAL_DATASET_NAME)    # HumanEval: benchmark for code generation

# EvalPlus datasets provide *harder test cases* for evaluation.
# These are optional — if they fail to load, the notebook still runs.
evalplus_mbpp_raw = load_dataset_safe(EVALPLUS_MBPP_NAME)

# ------------------------------------------------------------
# Safety checks: Ensure critical datasets loaded successfully.
# MBPP and HumanEval are required for training/evaluation.
# If they fail, raise RuntimeError immediately to stop execution.
# ------------------------------------------------------------
if mbpp_raw is None:
    raise RuntimeError("Could not load MBPP dataset.")

if humaneval_raw is None:
    raise RuntimeError("Could not load HumanEval dataset.")

# ------------------------------------------------------------
# Print dataset objects for quick inspection.
# This helps confirm structure (DatasetDict, Dataset, etc.)
# and ensures splits (train/test/validation) are available.
# ------------------------------------------------------------
print("MBPP:", mbpp_raw)
print("HumanEval:", humaneval_raw)
print("EvalPlus MBPP+:", evalplus_mbpp_raw)


Loading dataset: mbpp
Loading dataset: openai_humaneval
Loading dataset: evalplus/mbppplus
MBPP: DatasetDict({
    train: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 374
    })
    test: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 90
    })
    prompt: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 10
    })
})
HumanEval: DatasetDict({
    test: Dataset({
        features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
        num_rows: 164
    })
})
EvalPlus MBPP+: DatasetDict({
    test: Dataset({
        features: ['task_id', 'code', 'prompt', 'source_file',

In [51]:
# ============================================================
# CODE BLOCK 5: Normalization helpers
# ============================================================

def clean_code_text(code):
    if code is None:
        return ""

    code = str(code)
    code = code.replace("```python", "")
    code = code.replace("```", "")
    code = textwrap.dedent(code).strip()

    return code


def clean_instruction_text(text):
    if text is None:
        return ""

    text = str(text)
    text = " ".join(text.split())

    return text.strip()


def is_valid_python_syntax(code):
    try:
        ast.parse(code)
        return True
    except Exception:
        return False


def extract_function_name_from_text(text):
    match = re.search(
        r"def\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(",
        str(text)
    )

    if match:
        return match.group(1)

    return ""


def normalize_mbpp(example):
    return {
        "source_dataset": "MBPP",
        "task_id": str(example.get("task_id", "")),
        "instruction": clean_instruction_text(example.get("text", "")),
        "reference_code": clean_code_text(example.get("code", "")),
        "tests": example.get("test_list", []),
        "entry_point": ""
    }


def normalize_humaneval(example):
    prompt = example.get("prompt", "")
    canonical_solution = example.get("canonical_solution", "")
    tests = example.get("test", "")
    entry_point = example.get("entry_point", "")

    if not entry_point:
        entry_point = extract_function_name_from_text(prompt)

    full_reference_code = clean_code_text(prompt + canonical_solution)

    full_tests = tests

    if entry_point and isinstance(full_tests, str):
        full_tests = full_tests + f"\ncheck({entry_point})"

    return {
        "source_dataset": "HumanEval",
        "task_id": str(example.get("task_id", "")),
        "instruction": clean_code_text(prompt),
        "reference_code": full_reference_code,
        "tests": full_tests,
        "entry_point": entry_point
    }


def normalize_evalplus_mbpp(example):
    prompt = example.get("prompt", "")
    canonical_solution = example.get("canonical_solution", "")
    tests = example.get("test", "")
    task_id = example.get("task_id", example.get("problem_id", ""))
    entry_point = example.get("entry_point", "")

    if not entry_point:
        entry_point = extract_function_name_from_text(prompt)

    full_reference_code = clean_code_text(prompt + canonical_solution)

    return {
        "source_dataset": "EvalPlus-MBPPPlus",
        "task_id": str(task_id),
        "instruction": clean_code_text(prompt),
        "reference_code": full_reference_code,
        "tests": tests,
        "entry_point": entry_point
    }

In [52]:
# ============================================================
# CODE BLOCK 6: Normalize datasets
# ============================================================

# ------------------------------------------------------------
# Normalize MBPP dataset
# ------------------------------------------------------------
# Create a new DatasetDict to hold normalized MBPP splits.
mbpp = DatasetDict()

# Iterate over each split (train/test/validation) in the raw MBPP dataset.
# Apply the normalize_mbpp function to transform examples into a consistent schema.
# Remove original columns after mapping to keep only normalized fields.
for split in mbpp_raw.keys():
    mbpp[split] = mbpp_raw[split].map(
        normalize_mbpp,
        remove_columns=mbpp_raw[split].column_names
    )

# ------------------------------------------------------------
# Normalize HumanEval dataset
# ------------------------------------------------------------
humaneval = DatasetDict()

# Iterate over each split in the raw HumanEval dataset.
# Apply normalize_humaneval to unify schema (instruction, reference_code, tests).
for split in humaneval_raw.keys():
    humaneval[split] = humaneval_raw[split].map(
        normalize_humaneval,
        remove_columns=humaneval_raw[split].column_names
    )

# ------------------------------------------------------------
# Normalize EvalPlus MBPP+ dataset (optional)
# ------------------------------------------------------------
evalplus_mbpp = None

# Only normalize if dataset was successfully loaded.
if evalplus_mbpp_raw is not None:
    evalplus_mbpp = DatasetDict()

    # Iterate over each split in EvalPlus MBPP+ dataset.
    # Apply normalize_evalplus_mbpp to unify schema.
    for split in evalplus_mbpp_raw.keys():
        evalplus_mbpp[split] = evalplus_mbpp_raw[split].map(
            normalize_evalplus_mbpp,
            remove_columns=evalplus_mbpp_raw[split].column_names
        )

# ------------------------------------------------------------
# Print normalized datasets for inspection
# ------------------------------------------------------------
# This confirms that normalization worked and shows dataset structure.
print("Normalized MBPP:", mbpp)
print("Normalized HumanEval:", humaneval)
print("Normalized EvalPlus MBPP+:", evalplus_mbpp)


Normalized MBPP: DatasetDict({
    train: Dataset({
        features: ['task_id', 'source_dataset', 'instruction', 'reference_code', 'tests', 'entry_point'],
        num_rows: 374
    })
    test: Dataset({
        features: ['task_id', 'source_dataset', 'instruction', 'reference_code', 'tests', 'entry_point'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['task_id', 'source_dataset', 'instruction', 'reference_code', 'tests', 'entry_point'],
        num_rows: 90
    })
    prompt: Dataset({
        features: ['task_id', 'source_dataset', 'instruction', 'reference_code', 'tests', 'entry_point'],
        num_rows: 10
    })
})
Normalized HumanEval: DatasetDict({
    test: Dataset({
        features: ['task_id', 'entry_point', 'source_dataset', 'instruction', 'reference_code', 'tests'],
        num_rows: 164
    })
})
Normalized EvalPlus MBPP+: DatasetDict({
    test: Dataset({
        features: ['task_id', 'source_dataset', 'instruction', 'reference_code', 'te

In [53]:
# ============================================================
# CODE BLOCK 7: Dataset quality filtering
# ============================================================

def valid_stage2_example(example, require_valid_python=True):
    """
    Checks whether a dataset example meets quality criteria:
    - Instruction text must be non-empty and within word limits.
    - Reference code must be non-empty and within character limits.
    - Optionally validates Python syntax using ast.parse.
    """

    # Clean instruction and code text
    instruction = clean_instruction_text(example["instruction"])
    code = clean_code_text(example["reference_code"])

    # Reject empty instruction or code
    if instruction == "":
        return False
    if code == "":
        return False

    # Word count constraints for instruction
    instruction_words = len(instruction.split())
    if instruction_words < MIN_INSTRUCTION_WORDS:
        return False
    if instruction_words > MAX_INSTRUCTION_WORDS:
        return False

    # Character length constraints for code
    if len(code) < MIN_CODE_CHARS:
        return False
    if len(code) > MAX_CODE_CHARS:
        return False

    # Optional: validate Python syntax
    if require_valid_python and not is_valid_python_syntax(code):
        return False

    return True


def clean_stage2_dataset(ds, dataset_name, require_valid_python=True):
    """
    Cleans a dataset by:
    - Filtering out invalid examples using valid_stage2_example.
    - Removing duplicate instructions (case-insensitive).
    - Printing before/after counts for transparency.
    """

    before = len(ds)

    # Filter dataset using quality criteria
    ds = ds.filter(
        lambda ex: valid_stage2_example(
            ex,
            require_valid_python=require_valid_python
        )
    )

    # Deduplicate by instruction text (case-insensitive)
    seen = set()
    keep_indices = []
    for i, row in enumerate(ds):
        key = row["instruction"].lower().strip()
        if key not in seen:
            seen.add(key)
            keep_indices.append(i)

    # Select only unique examples
    ds = ds.select(keep_indices)

    after = len(ds)
    print(f"{dataset_name}: {before} -> {after} examples kept")

    return ds


# ------------------------------------------------------------
# Apply cleaning to MBPP dataset
# ------------------------------------------------------------
for split in mbpp.keys():
    mbpp[split] = clean_stage2_dataset(
        mbpp[split],
        f"MBPP-{split}",
        require_valid_python=True
    )

# ------------------------------------------------------------
# Apply cleaning to HumanEval dataset
# ------------------------------------------------------------
for split in humaneval.keys():
    humaneval[split] = clean_stage2_dataset(
        humaneval[split],
        f"HumanEval-{split}",
        require_valid_python=True
    )

# ------------------------------------------------------------
# Apply cleaning to EvalPlus MBPP+ dataset (if available)
# ------------------------------------------------------------
if evalplus_mbpp is not None:
    for split in evalplus_mbpp.keys():
        evalplus_mbpp[split] = clean_stage2_dataset(
            evalplus_mbpp[split],
            f"EvalPlus-MBPPPlus-{split}",
            require_valid_python=True
        )


MBPP-train: 374 -> 374 examples kept
MBPP-test: 500 -> 499 examples kept
MBPP-validation: 90 -> 90 examples kept
MBPP-prompt: 10 -> 10 examples kept
HumanEval-test: 164 -> 163 examples kept
EvalPlus-MBPPPlus-test: 378 -> 0 examples kept


In [54]:
# ============================================================
# CODE BLOCK 8: Train / validation / test split strategy
# ============================================================

def get_split_or_first(dataset_dict, preferred_split):
    if preferred_split in dataset_dict:
        return dataset_dict[preferred_split]

    first_split = list(dataset_dict.keys())[0]
    print(f"Using split '{first_split}' because '{preferred_split}' was not found.")

    return dataset_dict[first_split]


def limit_dataset(ds, limit, name):
    if limit is None:
        print(f"{name}: using full dataset ({len(ds)} examples)")
        return ds

    limit = min(limit, len(ds))
    print(f"{name}: using {limit}/{len(ds)} examples")

    return ds.select(range(limit))


def force_common_schema(ds):
    rows = []

    for example in ds:
        tests = example.get("tests", [])

        if tests is None:
            tests = []

        if not isinstance(tests, list):
            tests = [tests]

        tests = [str(t) for t in tests if t is not None]

        rows.append({
            "source_dataset": str(example.get("source_dataset", "")),
            "task_id": str(example.get("task_id", "")),
            "instruction": str(example.get("instruction", "")),
            "reference_code": str(example.get("reference_code", "")),
            "tests_json": json.dumps(tests),
            "entry_point": str(example.get("entry_point", ""))
        })

    return Dataset.from_list(rows)


# ------------------------------------------------------------
# MBPP split handling
# ------------------------------------------------------------

if all(k in mbpp for k in ["train", "validation", "test"]):
    mbpp_train = mbpp["train"]
    mbpp_validation = mbpp["validation"]
    mbpp_test = mbpp["test"]

elif all(k in mbpp for k in ["train", "test"]):
    temp_split = mbpp["train"].train_test_split(
        test_size=0.1,
        seed=SEED
    )

    mbpp_train = temp_split["train"]
    mbpp_validation = temp_split["test"]
    mbpp_test = mbpp["test"]

else:
    only_split = mbpp[list(mbpp.keys())[0]]

    temp = only_split.train_test_split(
        test_size=0.2,
        seed=SEED
    )

    mbpp_train = temp["train"]

    temp2 = temp["test"].train_test_split(
        test_size=0.5,
        seed=SEED
    )

    mbpp_validation = temp2["train"]
    mbpp_test = temp2["test"]


# ------------------------------------------------------------
# Evaluation-only datasets
# ------------------------------------------------------------

humaneval_test = get_split_or_first(humaneval, "test")

evalplus_mbpp_test = None

if evalplus_mbpp is not None:
    evalplus_mbpp_test = get_split_or_first(evalplus_mbpp, "test")


# ------------------------------------------------------------
# Apply practical limits
# ------------------------------------------------------------

mbpp_train = limit_dataset(
    mbpp_train.shuffle(seed=SEED),
    MBPP_TRAIN_EXAMPLES,
    "MBPP train"
)

mbpp_validation = limit_dataset(
    mbpp_validation.shuffle(seed=SEED),
    VALIDATION_EXAMPLES,
    "MBPP validation"
)

mbpp_test_data = limit_dataset(
    mbpp_test.shuffle(seed=SEED),
    TEST_EXAMPLES,
    "MBPP test"
)

humaneval_test_data = limit_dataset(
    humaneval_test.shuffle(seed=SEED),
    TEST_EXAMPLES,
    "HumanEval test"
)

evalplus_mbpp_test_data = None

if evalplus_mbpp_test is not None:
    evalplus_mbpp_test_data = limit_dataset(
        evalplus_mbpp_test.shuffle(seed=SEED),
        TEST_EXAMPLES,
        "EvalPlus MBPP+ test"
    )


# ------------------------------------------------------------
# Schema alignment
# ------------------------------------------------------------

train_data = force_common_schema(mbpp_train)
validation_data = force_common_schema(mbpp_validation)
mbpp_test_data = force_common_schema(mbpp_test_data)
humaneval_test_data = force_common_schema(humaneval_test_data)

if evalplus_mbpp_test_data is not None:
    evalplus_mbpp_test_data = force_common_schema(evalplus_mbpp_test_data)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\nFinal dataset sizes:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("MBPP Test:", len(mbpp_test_data))
print("HumanEval Test:", len(humaneval_test_data))

if evalplus_mbpp_test_data is not None:
    print("EvalPlus MBPP+ Test:", len(evalplus_mbpp_test_data))

print("\nTrain source distribution:")
print(pd.Series(train_data["source_dataset"]).value_counts())

print("\nValidation source distribution:")
print(pd.Series(validation_data["source_dataset"]).value_counts())

MBPP train: using full dataset (374 examples)
MBPP validation: using full dataset (90 examples)
MBPP test: using 30/499 examples
HumanEval test: using 30/163 examples
EvalPlus MBPP+ test: using 0/0 examples

Final dataset sizes:
Train: 374
Validation: 90
MBPP Test: 30
HumanEval Test: 30
EvalPlus MBPP+ Test: 0

Train source distribution:
MBPP    374
Name: count, dtype: int64

Validation source distribution:
MBPP    90
Name: count, dtype: int64


In [56]:
# ============================================================
# CODE BLOCK 8B: HumanEval-safe prompt construction helpers
# ============================================================

def extract_required_names_from_tests(tests_json):
    if tests_json is None:
        return []

    if isinstance(tests_json, str):
        try:
            tests = json.loads(tests_json)
        except Exception:
            tests = [tests_json]
    elif isinstance(tests_json, list):
        tests = tests_json
    else:
        tests = [str(tests_json)]

    names = []

    for test in tests:
        test = str(test)

        matches = re.findall(
            r"assert\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(",
            test
        )

        for name in matches:
            if name not in names:
                names.append(name)

    return names


def get_required_signature(example):
    source_dataset = example.get("source_dataset", "")
    entry_point = example.get("entry_point", "")
    instruction = example.get("instruction", "")

    if source_dataset == "HumanEval":
        if not entry_point:
            entry_point = extract_function_name_from_text(instruction)

        if entry_point:
            return f"\nRequired function name: {entry_point}."

    names = extract_required_names_from_tests(
        example.get("tests_json", "")
    )

    if len(names) == 1:
        return f"\nRequired function name: {names[0]}."

    if len(names) > 1:
        return "\nRequired function names: " + ", ".join(names) + "."

    return ""


def build_prompt(example):
    source_dataset = example.get("source_dataset", "")
    instruction = example.get("instruction", "")
    signature_hint = get_required_signature(example)

    if source_dataset == "HumanEval":
        return f"""<|im_start|>system
You are RepoCoder Studio, an expert Python coding assistant.
Complete the given Python function exactly.
Do not rename the function.
Return only valid Python code.
Do not include metadata, JSON, tests, explanations, markdown, or comments.
<|im_end|>
<|im_start|>user
Complete this exact Python function and return the full function implementation:

{instruction}
{signature_hint}
<|im_end|>
<|im_start|>assistant
"""

    return f"""<|im_start|>system
You are RepoCoder Studio, an expert Python coding assistant.
Write only valid Python code.
Do not include metadata, JSON, tests, explanations, markdown, or comments.
Define only the required function(s).
<|im_end|>
<|im_start|>user
Write Python code for this task:

{clean_instruction_text(instruction)}
{signature_hint}
<|im_end|>
<|im_start|>assistant
"""


def build_training_text(example):
    return (
        build_prompt(example)
        + clean_code_text(example["reference_code"]).strip()
        + "\n<|im_end|>"
    )

In [57]:
# ============================================================
# CODE BLOCK 9: Add prompt fields
# ============================================================

def add_prompt_fields(example):
    example["prompt"] = build_prompt(example)
    example["training_text"] = build_training_text(example)
    return example


train_data = train_data.map(add_prompt_fields)
validation_data = validation_data.map(add_prompt_fields)
mbpp_test_data = mbpp_test_data.map(add_prompt_fields)
humaneval_test_data = humaneval_test_data.map(add_prompt_fields)

if evalplus_mbpp_test_data is not None and len(evalplus_mbpp_test_data) > 0:
    evalplus_mbpp_test_data = evalplus_mbpp_test_data.map(add_prompt_fields)

print("Sample MBPP prompt:")
print(mbpp_test_data[0]["prompt"][:1200])

print("\nSample HumanEval prompt:")
print(humaneval_test_data[0]["prompt"][:1200])

Map:   0%|          | 0/374 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Sample MBPP prompt:
<|im_start|>system
You are RepoCoder Studio, an expert Python coding assistant.
Write only valid Python code.
Do not include metadata, JSON, tests, explanations, markdown, or comments.
Define only the required function(s).
<|im_end|>
<|im_start|>user
Write Python code for this task:

Write a function to find the largest palindromic number in the given array.

Required function name: largest_palindrome.
<|im_end|>
<|im_start|>assistant


Sample HumanEval prompt:
<|im_start|>system
You are RepoCoder Studio, an expert Python coding assistant.
Complete the given Python function exactly.
Do not rename the function.
Return only valid Python code.
Do not include metadata, JSON, tests, explanations, markdown, or comments.
<|im_end|>
<|im_start|>user
Complete this exact Python function and return the full function implementation:

def remove_vowels(text):
    """
    remove_vowels is a function that takes string and returns string without vowels.
    >>> remove_vowels('')
  

In [58]:
# ============================================================
# CODE BLOCK 9B: Prompt contamination sanity check
# ============================================================

bad_terms = [
    "source_dataset",
    "task_id",
    "tests_json",
    "reference_code",
    "METADATA",
    "expected_output"
]

for name, ds in [
    ("MBPP", mbpp_test_data),
    ("HumanEval", humaneval_test_data)
]:
    prompt = ds[0]["prompt"]
    found = [term for term in bad_terms if term in prompt]

    print(name, "bad prompt terms:", found)

    if found:
        raise RuntimeError(f"{name} prompt is contaminated: {found}")

print("Prompts are clean.")

MBPP bad prompt terms: []
HumanEval bad prompt terms: []
Prompts are clean.


In [59]:
# ============================================================
# CODE BLOCK 10: Model loading utilities
# ============================================================

def load_model_and_tokenizer(model_name):
    """
    Loads a Hugging Face causal language model and its tokenizer.
    Steps:
    1. Print which model is being loaded (for logging/debugging).
    2. Load tokenizer from pretrained model.
       - If pad_token is missing, set it to eos_token (common fix).
    3. Load model from pretrained weights.
       - Uses float16 if GPU is available, otherwise float32.
       - device_map="auto" automatically places model layers across devices.
    4. Set pad_token_id in model config to match tokenizer.
    5. Put model in evaluation mode (no training).
    Returns: (model, tokenizer)
    """
    print("Loading:", model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Ensure pad_token exists (important for batching/padding sequences)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # Sync pad_token_id between tokenizer and model
    model.config.pad_token_id = tokenizer.pad_token_id

    # Set model to evaluation mode (disables dropout, etc.)
    model.eval()
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    model.generation_config.do_sample = False

    return model, tokenizer


def unload_model(model=None, tokenizer=None):
    """
    Safely unloads a model and tokenizer from memory.
    Steps:
    1. Delete model and tokenizer objects (if they exist).
    2. Run garbage collection to free Python memory.
    3. If GPU is available, clear CUDA cache to free VRAM.
    4. Print confirmation message.
    """
    try:
        del model
    except Exception:
        pass

    try:
        del tokenizer
    except Exception:
        pass

    # Force garbage collection
    gc.collect()

    # Clear GPU memory if available
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Model unloaded.")


In [60]:
# ============================================================
# CODE BLOCK 11: Tokenization with assistant-only loss masking
# ============================================================

def tokenize_training_example(example):
    full_text = example["training_text"]
    prompt_text = example["prompt"]

    tokenized_full = qwen_tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_TOTAL_LENGTH,
        padding="max_length"
    )

    tokenized_prompt = qwen_tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_TOTAL_LENGTH,
        padding=False
    )

    labels = tokenized_full["input_ids"].copy()
    prompt_len = len(tokenized_prompt["input_ids"])

    # Mask prompt tokens
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100

    # Mask padding tokens
    labels = [
        -100 if token_id == qwen_tokenizer.pad_token_id else label
        for token_id, label in zip(tokenized_full["input_ids"], labels)
    ]

    tokenized_full["labels"] = labels
    return tokenized_full


def tokenize_eval_example(example):
    return tokenize_training_example(example)


qwen_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_NAME,
    trust_remote_code=True
)

if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

tokenized_train = train_data.map(
    tokenize_training_example,
    remove_columns=train_data.column_names
)

tokenized_validation = validation_data.map(
    tokenize_eval_example,
    remove_columns=validation_data.column_names
)

tokenization_stats = {
    "train_examples": len(tokenized_train),
    "validation_examples": len(tokenized_validation),
    "max_total_length": MAX_TOTAL_LENGTH,
    "loss_masking": "Prompt tokens masked; only assistant solution tokens supervised."
}

print(tokenization_stats)

Map:   0%|          | 0/374 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

{'train_examples': 374, 'validation_examples': 90, 'max_total_length': 768, 'loss_masking': 'Prompt tokens masked; only assistant solution tokens supervised.'}


In [67]:
# ============================================================
# CODE BLOCK 12: Generation, extraction, and execution helpers
# ============================================================

def strip_prompt_artifacts(text):
    if text is None:
        return ""

    text = str(text)

    bad_markers = [
        "<|im_start|>",
        "<|im_end|>",
        "<|system|>",
        "<|user|>",
        "<|assistant|>",
        "<|end|>",
        "<|endoftext|>",
        "```python",
        "```",
        "### Instruction:",
        "### Response:"
    ]

    for marker in bad_markers:
        text = text.replace(marker, "")

    return text.strip()


def extract_python_code(generated_text, entry_point=""):
    text = strip_prompt_artifacts(generated_text)

    text = text.replace("０", "0")
    text = text.replace("：", ":")

    # Remove common leaked metadata before code
    metadata_markers = [
        "source_dataset",
        "task_id",
        "tests_json",
        "reference_code",
        "expected_output",
        "METADATA"
    ]

    if any(marker in text for marker in metadata_markers):
        if entry_point:
            idx = text.find(f"def {entry_point}")
            if idx != -1:
                text = text[idx:]

    # Prefer exact HumanEval entry point if available
    if entry_point:
        idx = text.find(f"def {entry_point}")
        if idx != -1:
            text = text[idx:].strip()

    # Otherwise start from first Python construct
    if not text.strip().startswith(("def ", "class ", "from ", "import ")):
        starts = []

        for marker in [
            "\ndef ",
            "\nclass ",
            "\nfrom ",
            "\nimport ",
            "def ",
            "class ",
            "from ",
            "import "
        ]:
            idx = text.find(marker)

            if idx != -1:
                starts.append(idx)

        if starts:
            text = text[min(starts):].strip()

    stop_markers = [
        "\n# Test",
        "\n# test",
        "\nassert ",
        "\nif __name__",
        "\nimport unittest",
        "\nclass Test",
        "\nExpected",
        "\nActual",
        "\nPASS",
        "\nCorrect:",
        "\nError:",
        "\nExplanation:",
        "\nThe function",
        "\nThis function",
        "\n# Example",
        "\n# example",
        "\nExample usage",
        "\nexample usage",
        "\nprint(",
        "\nOutput:"
    ]

    stop_positions = []

    for marker in stop_markers:
        pos = text.find(marker)

        if pos != -1:
            stop_positions.append(pos)

    if stop_positions:
        text = text[:min(stop_positions)].strip()

    return text.strip()


def generate_code(
    model,
    tokenizer,
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    entry_point=""
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "num_return_sequences": 1,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id
    }

    if do_sample:
        generation_kwargs["temperature"] = 0.2
        generation_kwargs["top_p"] = 0.95

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **generation_kwargs
        )

    new_tokens = output_ids[0][input_len:]

    decoded = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return extract_python_code(
        decoded,
        entry_point=entry_point
    )


def run_python_tests(generated_code, tests_json, timeout=5, entry_point="", prompt_prefix=""):
    """
    Executes generated Python code against benchmark tests.

    HumanEval fix:
    - Some HumanEval prompts include helper functions before the target function.
    - Therefore, when prompt_prefix is provided, execute prompt_prefix + generated completion.
    """

    generated_code = extract_python_code(
        generated_code,
        entry_point=entry_point
    )

    if prompt_prefix:
        executable_code = clean_code_text(prompt_prefix) + "\n" + generated_code
    else:
        executable_code = generated_code

    if executable_code.strip() == "":
        return {
            "passed": False,
            "stderr": "Empty generated code"
        }

    try:
        ast.parse(executable_code)
    except Exception:
        return {
            "passed": False,
            "stderr": "Generated code has invalid Python syntax"
        }

    if isinstance(tests_json, str):
        try:
            tests = json.loads(tests_json)
        except Exception:
            tests = [tests_json]
    elif isinstance(tests_json, list):
        tests = tests_json
    else:
        tests = [str(tests_json)]

    test_code = executable_code + "\n\n" + "\n".join([str(t) for t in tests])

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(test_code)
        temp_path = f.name

    try:
        result = subprocess.run(
            ["python", temp_path],
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return {
            "passed": result.returncode == 0,
            "stderr": result.stderr
        }

    except subprocess.TimeoutExpired:
        return {
            "passed": False,
            "stderr": "Execution timed out"
        }

    finally:
        try:
            os.remove(temp_path)
        except Exception:
            pass

In [68]:
# ============================================================
# CODE BLOCK 13: Model evaluation harness
# ============================================================

def estimate_pass_at_k(correctness, k):
    """
    Computes pass@k for a list of boolean generation correctness values.
    For this demo setup, correctness is computed directly from generated samples.
    """
    if len(correctness) == 0:
        return np.nan

    k = min(k, len(correctness))

    return float(any(correctness[:k]))


def evaluate_model_on_dataset(model, tokenizer, dataset, dataset_name, model_label):
    """
    Evaluates model on MBPP / HumanEval-style executable Python tasks.

    For each example:
    - Generates NUM_GENERATIONS candidate solutions.
    - Extracts clean Python code.
    - Executes generated code against dataset tests.
    - Computes pass@k and execution accuracy.

    HumanEval-specific fix:
    - Passes entry_point into generation and testing so extraction prefers
      the exact required function name, e.g. remove_vowels.
    """

    detailed_rows = []
    summary_rows = []

    for example in tqdm(dataset, desc=f"{model_label} on {dataset_name}"):

        generations = []
        correctness = []
        execution_errors = []

        entry_point = example.get("entry_point", "")

        for gen_idx in range(NUM_GENERATIONS):

            generated_code = generate_code(
                model=model,
                tokenizer=tokenizer,
                prompt=example["prompt"],
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=(gen_idx > 0),
                entry_point=entry_point
            )

            prompt_prefix = ""

            if example.get("source_dataset", "") == "HumanEval":
                prompt_prefix = example.get("instruction", "")

            execution_result = run_python_tests(
                generated_code=generated_code,
                tests_json=example["tests_json"],
                timeout=5,
                entry_point=entry_point,
                prompt_prefix=prompt_prefix
            )

            generations.append(generated_code)
            correctness.append(bool(execution_result["passed"]))
            execution_errors.append(execution_result["stderr"])

        pass_scores = {}

        for k in PASS_K_VALUES:
            pass_scores[f"pass@{k}"] = estimate_pass_at_k(
                correctness,
                k
            )

        execution_accuracy = float(np.mean(correctness)) if len(correctness) > 0 else np.nan

        row = {
            "model": model_label,
            "dataset": dataset_name,
            "task_id": example.get("task_id", ""),
            "source_dataset": example.get("source_dataset", ""),
            "entry_point": entry_point,
            "instruction": example.get("instruction", ""),
            "reference_code": example.get("reference_code", ""),
            "tests_json": example.get("tests_json", ""),
            "generations": generations,
            "correctness": correctness,
            "execution_errors": execution_errors,
            "execution_accuracy": execution_accuracy,
            **pass_scores
        }

        detailed_rows.append(row)

    summary = {
        "model": model_label,
        "dataset": dataset_name,
        "num_examples": len(detailed_rows),
        "execution_accuracy": float(np.mean([
            row["execution_accuracy"] for row in detailed_rows
        ])) if detailed_rows else np.nan,
        "avg_codebleu": None
    }

    for k in PASS_K_VALUES:
        metric_name = f"pass@{k}"

        summary[metric_name] = float(np.mean([
            row[metric_name] for row in detailed_rows
        ])) if detailed_rows else np.nan

    # Reorder summary so pass@k appears before execution accuracy
    ordered_summary = {
        "model": summary["model"],
        "dataset": summary["dataset"],
        "num_examples": summary["num_examples"]
    }

    for k in PASS_K_VALUES:
        ordered_summary[f"pass@{k}"] = summary[f"pass@{k}"]

    ordered_summary["execution_accuracy"] = summary["execution_accuracy"]
    ordered_summary["avg_codebleu"] = summary["avg_codebleu"]

    return detailed_rows, ordered_summary

In [69]:
# ============================================================
# CODE BLOCK 14: Evaluate Qwen pretrained baseline
# ============================================================

# ------------------------------------------------------------
# Initialize result containers
# ------------------------------------------------------------
# all_results: stores detailed evaluation outputs (per dataset).
# all_summaries: stores high-level summaries (aggregated metrics).
all_results = []
all_summaries = []

# ------------------------------------------------------------
# Run evaluation only if flag is enabled
# ------------------------------------------------------------
if RUN_QWEN_BASELINE:
    # --------------------------------------------------------
    # Load Qwen pretrained model and tokenizer
    # --------------------------------------------------------
    # This loads the Hugging Face model and tokenizer specified
    # by QWEN_MODEL_NAME. The model is used for inference,
    # and the tokenizer prepares text inputs for the model.
    qwen_base_model, qwen_base_tokenizer = load_model_and_tokenizer(
        QWEN_MODEL_NAME
    )

    # --------------------------------------------------------
    # Evaluate on MBPP test set
    # --------------------------------------------------------
    # Produces detailed results and a summary dictionary.
    qwen_mbpp_results, qwen_mbpp_summary = evaluate_model_on_dataset(
        qwen_base_model,
        qwen_base_tokenizer,
        mbpp_test_data,       # dataset
        "MBPP-Test",          # dataset label
        "Qwen-Pretrained"     # model label
    )

    # --------------------------------------------------------
    # Evaluate on HumanEval test set
    # --------------------------------------------------------
    qwen_humaneval_results, qwen_humaneval_summary = evaluate_model_on_dataset(
        qwen_base_model,
        qwen_base_tokenizer,
        humaneval_test_data,
        "HumanEval",
        "Qwen-Pretrained"
    )

    # --------------------------------------------------------
    # Collect results and summaries
    # --------------------------------------------------------
    all_results.extend([
        qwen_mbpp_results,
        qwen_humaneval_results
    ])

    all_summaries.extend([
        qwen_mbpp_summary,
        qwen_humaneval_summary
    ])

    # --------------------------------------------------------
    # Optional: Evaluate on EvalPlus MBPP+ (harder test cases)
    # --------------------------------------------------------
    if evalplus_mbpp_test_data is not None and len(evalplus_mbpp_test_data) > 0:
        qwen_evalplus_mbpp_results, qwen_evalplus_mbpp_summary = evaluate_model_on_dataset(
            qwen_base_model,
            qwen_base_tokenizer,
            evalplus_mbpp_test_data,
            "EvalPlus-MBPPPlus",
            "Qwen-Pretrained"
        )

        # Append EvalPlus results to global containers
        all_results.append(qwen_evalplus_mbpp_results)
        all_summaries.append(qwen_evalplus_mbpp_summary)

    # --------------------------------------------------------
    # Print summaries for Qwen pretrained baseline
    # --------------------------------------------------------
    print("Qwen pretrained summaries:")

    # Iterate through all summaries and pretty-print JSON
    # Only print those belonging to "Qwen-Pretrained"
    for summary in all_summaries:
        if summary["model"] == "Qwen-Pretrained":
            print(json.dumps(summary, indent=2))

    # --------------------------------------------------------
    # Unload model and tokenizer to free memory
    # --------------------------------------------------------
    # This is important in GPU environments to avoid OOM errors
    # when loading other models later.
    unload_model(qwen_base_model, qwen_base_tokenizer)


Loading: Qwen/Qwen2.5-Coder-0.5B-Instruct


Qwen-Pretrained on MBPP-Test:   0%|          | 0/30 [00:00<?, ?it/s]

Qwen-Pretrained on HumanEval:   0%|          | 0/30 [00:00<?, ?it/s]

Qwen pretrained summaries:
{
  "model": "Qwen-Pretrained",
  "dataset": "MBPP-Test",
  "num_examples": 30,
  "pass@1": 0.5333333333333333,
  "pass@3": 0.6,
  "execution_accuracy": 0.4888888888888889,
  "avg_codebleu": null
}
{
  "model": "Qwen-Pretrained",
  "dataset": "HumanEval",
  "num_examples": 30,
  "pass@1": 0.36666666666666664,
  "pass@3": 0.4,
  "execution_accuracy": 0.3555555555555555,
  "avg_codebleu": null
}
Model unloaded.


In [70]:
# ============================================================
# CODE BLOCK 14B: Inspect baseline generations
# ============================================================

def inspect_generation_results(results, n=3):
    """
    Prints representative model generations and execution outcomes.
    Compatible with both old and new result schemas.
    """

    for row in results[:n]:
        print("=" * 100)
        print(f"DATASET: {row.get('dataset', '')}")
        print(f"TASK ID: {row.get('task_id', '')}")

        print("\nINSTRUCTION:")
        print(row.get("instruction", "")[:1200])

        print("\nREFERENCE CODE:")
        print(row.get("reference_code", "")[:1200])

        generations = row.get("generations", [])
        correctness = row.get("correctness", row.get("correct_list", []))
        errors = row.get("execution_errors", [])

        for j, gen in enumerate(generations):
            print("-" * 80)
            print(f"GENERATION {j + 1}:")
            print(str(gen)[:1200])

            if j < len(correctness):
                print("Correct:", correctness[j])
            else:
                print("Correct: N/A")

            if j < len(errors):
                print("Error:", errors[j])
            else:
                print("Error: N/A")

        print("=" * 100)


print("Inspecting Qwen MBPP baseline generations:")
inspect_generation_results(
    qwen_mbpp_results,
    n=3
)

print("\nInspecting Qwen HumanEval baseline generations:")
inspect_generation_results(
    qwen_humaneval_results,
    n=3
)

Inspecting Qwen MBPP baseline generations:
DATASET: MBPP-Test
TASK ID: 485

INSTRUCTION:
Write a function to find the largest palindromic number in the given array.

REFERENCE CODE:
def is_palindrome(n) : 
	divisor = 1
	while (n / divisor >= 10) : 
		divisor *= 10
	while (n != 0) : 
		leading = n // divisor 
		trailing = n % 10
		if (leading != trailing) : 
			return False
		n = (n % divisor) // 10
		divisor = divisor // 100
	return True
def largest_palindrome(A, n) : 
	A.sort() 
	for i in range(n - 1, -1, -1) : 
		if (is_palindrome(A[i])) : 
			return A[i] 
	return -1
--------------------------------------------------------------------------------
GENERATION 1:
def largest_palindrome(arr):
    """
    Finds the largest palindrome in the given array of integers.

    Args:
    arr (list): A list of integers.

    Returns:
    int: The largest palindrome found in the array.
    """
    # Sort the array in descending order
    arr.sort(reverse=True)
    
    # Iterate through the sorted 

In [71]:
# ============================================================
# CODE BLOCK 15: Evaluate DeepSeek pretrained baseline
# ============================================================

# ------------------------------------------------------------
# Run evaluation only if flag is enabled
# ------------------------------------------------------------
if RUN_DEEPSEEK_BASELINE:
    # --------------------------------------------------------
    # Load DeepSeek pretrained model and tokenizer
    # --------------------------------------------------------
    # This loads the Hugging Face model and tokenizer specified
    # by DEEPSEEK_MODEL_NAME. The model is used for inference,
    # and the tokenizer prepares text inputs for the model.
    deepseek_model, deepseek_tokenizer = load_model_and_tokenizer(
        DEEPSEEK_MODEL_NAME
    )

    # --------------------------------------------------------
    # Evaluate on MBPP test set
    # --------------------------------------------------------
    # Produces detailed results and a summary dictionary.
    deepseek_mbpp_results, deepseek_mbpp_summary = evaluate_model_on_dataset(
        deepseek_model,
        deepseek_tokenizer,
        mbpp_test_data,       # dataset
        "MBPP-Test",          # dataset label
        "DeepSeek-Pretrained" # model label
    )

    # --------------------------------------------------------
    # Evaluate on HumanEval test set
    # --------------------------------------------------------
    deepseek_humaneval_results, deepseek_humaneval_summary = evaluate_model_on_dataset(
        deepseek_model,
        deepseek_tokenizer,
        humaneval_test_data,
        "HumanEval",
        "DeepSeek-Pretrained"
    )

    # --------------------------------------------------------
    # Collect results and summaries
    # --------------------------------------------------------
    all_results.extend([
        deepseek_mbpp_results,
        deepseek_humaneval_results
    ])

    all_summaries.extend([
        deepseek_mbpp_summary,
        deepseek_humaneval_summary
    ])

    # --------------------------------------------------------
    # Optional: Evaluate on EvalPlus MBPP+ (harder test cases)
    # --------------------------------------------------------
    if evalplus_mbpp_test_data is not None and len(evalplus_mbpp_test_data) > 0:
        deepseek_evalplus_mbpp_results, deepseek_evalplus_mbpp_summary = evaluate_model_on_dataset(
            deepseek_model,
            deepseek_tokenizer,
            evalplus_mbpp_test_data,
            "EvalPlus-MBPPPlus",
            "DeepSeek-Pretrained"
        )

        # Append EvalPlus results to global containers
        all_results.append(deepseek_evalplus_mbpp_results)
        all_summaries.append(deepseek_evalplus_mbpp_summary)

    # --------------------------------------------------------
    # Print summaries for DeepSeek pretrained baseline
    # --------------------------------------------------------
    print("DeepSeek pretrained summaries:")

    # Iterate through all summaries and pretty-print JSON
    # Only print those belonging to "DeepSeek-Pretrained"
    for summary in all_summaries:
        if summary["model"] == "DeepSeek-Pretrained":
            print(json.dumps(summary, indent=2))

    # --------------------------------------------------------
    # Unload model and tokenizer to free memory
    # --------------------------------------------------------
    # This is important in GPU environments to avoid OOM errors
    # when loading other models later.
    unload_model(deepseek_model, deepseek_tokenizer)


In [72]:
# ============================================================
# CODE BLOCK 15B: Inspect DeepSeek Baseline Generations
# ============================================================
if RUN_DEEPSEEK_BASELINE:
  print("Inspecting DeepSeek MBPP baseline generations:")
  inspect_generation_results(
      deepseek_mbpp_results,
      n=3
  )

  print("\nInspecting DeepSeek HumanEval baseline generations:")
  inspect_generation_results(
      deepseek_humaneval_results,
      n=3
  )

In [73]:
# ============================================================
# CODE BLOCK 16: Load Qwen with LoRA for fine-tuning
# ============================================================

# Step 1: Load the base Qwen model and tokenizer
qwen_ft_model, qwen_ft_tokenizer = load_model_and_tokenizer(QWEN_MODEL_NAME)

# Step 2: Define LoRA configuration
# LoRA (Low-Rank Adaptation) injects trainable adapters into specific layers
# of the model, allowing efficient fine-tuning without updating all parameters.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # Task type: causal language modeling
    r=8,                            # Rank of the low-rank adapters (capacity vs efficiency)
    lora_alpha=16,                  # Scaling factor for updates
    lora_dropout=0.05,              # Dropout applied to adapter layers for regularization
    target_modules=[                # Layers where LoRA adapters will be injected
        "q_proj",                   # Attention query projection
        "k_proj",                   # Attention key projection
        "v_proj",                   # Attention value projection
        "o_proj",                   # Attention output projection
        "gate_proj",                # Feed-forward gating projection
        "up_proj",                  # Feed-forward up projection
        "down_proj"                 # Feed-forward down projection
    ],
    bias="none"                     # Bias parameters are not trained, only adapter weights
)

# Step 3: Wrap the base model with PEFT (LoRA)
# This injects LoRA adapters into the specified modules.
# The base model remains frozen; only adapter weights are trainable.
qwen_ft_model = get_peft_model(qwen_ft_model, lora_config)

# Step 4: Print trainable parameters
# Confirms how many parameters are trainable vs frozen.
# Typically, LoRA reduces trainable parameters by orders of magnitude compared to full fine-tuning.
qwen_ft_model.print_trainable_parameters()


Loading: Qwen/Qwen2.5-Coder-0.5B-Instruct
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [74]:
# ============================================================
# CODE BLOCK 17: Tokenization with code-only labels
# ============================================================

def tokenize_for_training(example):
    """
    Stage 2 training strategy:

    Input:
        Prompt + Reference Code

    Loss:
        ONLY on reference code tokens

    Prompt tokens:
        masked with -100 (ignored by loss)

    Padding:
        masked with -100 (ignored by loss)

    Mirrors Stage 1 docstring-only supervision.
    """

    # Separate prompt and code
    prompt_text = example["prompt"]
    code_text = example["reference_code"]

    # Concatenate prompt + code for full input
    full_text = prompt_text + code_text

    # Tokenize full text (prompt + code)
    tokenized_full = qwen_ft_tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_TOTAL_LENGTH
    )

    # Tokenize prompt alone (to measure prompt length)
    tokenized_prompt = qwen_ft_tokenizer(
        prompt_text,
        truncation=True,
        padding=False,
        max_length=MAX_TOTAL_LENGTH
    )

    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]

    # Copy input_ids to labels
    labels = input_ids.copy()

    # Prompt length (used to mask prompt tokens)
    prompt_length = len(tokenized_prompt["input_ids"])

    # Mask prompt tokens and padding with -100
    labels = [
        token_id
        if (index >= prompt_length and mask == 1)  # supervise only code tokens
        else -100                                 # ignore prompt/padding
        for index, (token_id, mask)
        in enumerate(zip(input_ids, attention_mask))
    ]

    tokenized_full["labels"] = labels

    return tokenized_full


# ------------------------------------------------------------
# Tokenize train
# ------------------------------------------------------------
tokenized_train = train_data.map(
    tokenize_for_training,
    remove_columns=train_data.column_names
)

# ------------------------------------------------------------
# Tokenize validation
# ------------------------------------------------------------
tokenized_validation = validation_data.map(
    tokenize_for_training,
    remove_columns=validation_data.column_names
)

print("Tokenization completed.")

print("Train keys:")
print(tokenized_train[0].keys())

print("\nValidation keys:")
print(tokenized_validation[0].keys())


# ============================================================
# SANITY CHECK 1
# ============================================================

# Inspect labels for a single validation example
sample_labels = tokenized_validation[0]["labels"]

# Count supervised vs masked tokens
num_supervised_tokens = sum(1 for label in sample_labels if label != -100)
num_masked_tokens = sum(1 for label in sample_labels if label == -100)

print("\nSingle-example sanity check:")
print("Supervised code tokens:", num_supervised_tokens)
print("Masked prompt/padding tokens:", num_masked_tokens)


# ============================================================
# SANITY CHECK 2
# ============================================================

# Inspect multiple validation examples
num_examples = min(20, len(tokenized_validation))

supervised_counts = []
masked_counts = []

for i in range(num_examples):
    labels = tokenized_validation[i]["labels"]

    supervised = sum(1 for label in labels if label != -100)
    masked = sum(1 for label in labels if label == -100)

    supervised_counts.append(supervised)
    masked_counts.append(masked)

# Aggregate sanity statistics
sanity_stats = {
    "checked_examples": num_examples,
    "avg_supervised_code_tokens": float(np.mean(supervised_counts)),
    "min_supervised_code_tokens": int(np.min(supervised_counts)),
    "max_supervised_code_tokens": int(np.max(supervised_counts)),
    "avg_masked_prompt_padding_tokens": float(np.mean(masked_counts))
}

print("\nTokenization sanity check:")
print(json.dumps(sanity_stats, indent=2))


Map:   0%|          | 0/374 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenization completed.
Train keys:
dict_keys(['input_ids', 'attention_mask', 'labels'])

Validation keys:
dict_keys(['input_ids', 'attention_mask', 'labels'])

Single-example sanity check:
Supervised code tokens: 22
Masked prompt/padding tokens: 746

Tokenization sanity check:
{
  "checked_examples": 20,
  "avg_supervised_code_tokens": 51.0,
  "min_supervised_code_tokens": 18,
  "max_supervised_code_tokens": 153,
  "avg_masked_prompt_padding_tokens": 717.0
}


In [75]:
# ============================================================
# CODE BLOCK 18: LoRA fine-tuning
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=2,
    save_steps=500,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    fp16=False,
    report_to="none",
    save_total_limit=2
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=qwen_ft_tokenizer,
    mlm=False
)

trainer = Trainer(
    model=qwen_ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    tokenizer=qwen_ft_tokenizer,
    data_collator=data_collator
)

if RUN_FINE_TUNING:
    trainer.train()

    final_model_dir = os.path.join(OUTPUT_DIR, "qwen_lora_finetuned")

    qwen_ft_model.save_pretrained(final_model_dir)
    qwen_ft_tokenizer.save_pretrained(final_model_dir)

    print("Fine-tuned Qwen LoRA adapter saved to:", final_model_dir)
else:
    print("Skipping fine-tuning.")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss


Fine-tuned Qwen LoRA adapter saved to: /content/RepoCoderStudio_Stage2_MBPP_SafeFT/qwen_lora_finetuned


In [76]:
# ============================================================
# CODE BLOCK 19: Evaluate fine-tuned Qwen
# ============================================================

# ------------------------------------------------------------
# Label for fine-tuned model
# ------------------------------------------------------------
# This label is used in summaries and results to distinguish
# fine-tuned Qwen runs from pretrained baselines.
FT_MODEL_LABEL = "Qwen-FineTuned-MBPP"

# ------------------------------------------------------------
# Evaluate on MBPP test set
# ------------------------------------------------------------
# Produces detailed results and a summary dictionary.
qwen_ft_mbpp_results, qwen_ft_mbpp_summary = evaluate_model_on_dataset(
    qwen_ft_model,
    qwen_ft_tokenizer,
    mbpp_test_data,       # dataset
    "MBPP-Test",          # dataset label
    FT_MODEL_LABEL        # model label
)

# ------------------------------------------------------------
# Evaluate on HumanEval test set
# ------------------------------------------------------------
qwen_ft_humaneval_results, qwen_ft_humaneval_summary = evaluate_model_on_dataset(
    qwen_ft_model,
    qwen_ft_tokenizer,
    humaneval_test_data,
    "HumanEval",
    FT_MODEL_LABEL
)

# ------------------------------------------------------------
# Collect results and summaries
# ------------------------------------------------------------
all_results.extend([
    qwen_ft_mbpp_results,
    qwen_ft_humaneval_results
])

all_summaries.extend([
    qwen_ft_mbpp_summary,
    qwen_ft_humaneval_summary
])

# ------------------------------------------------------------
# Optional: Evaluate on EvalPlus MBPP+ (harder test cases)
# ------------------------------------------------------------
if evalplus_mbpp_test_data is not None and len(evalplus_mbpp_test_data) > 0:
    qwen_ft_evalplus_mbpp_results, qwen_ft_evalplus_mbpp_summary = evaluate_model_on_dataset(
        qwen_ft_model,
        qwen_ft_tokenizer,
        evalplus_mbpp_test_data,
        "EvalPlus-MBPPPlus",
        FT_MODEL_LABEL
    )

    # Append EvalPlus results to global containers
    all_results.append(qwen_ft_evalplus_mbpp_results)
    all_summaries.append(qwen_ft_evalplus_mbpp_summary)

# ------------------------------------------------------------
# Print summaries for fine-tuned Qwen
# ------------------------------------------------------------
print("Fine-tuned Qwen summaries:")

# Iterate through all summaries and pretty-print JSON
# Only print those belonging to the fine-tuned Qwen label
for summary in all_summaries:
    if summary["model"] == FT_MODEL_LABEL:
        print(json.dumps(summary, indent=2))


Qwen-FineTuned-MBPP on MBPP-Test:   0%|          | 0/30 [00:00<?, ?it/s]

Qwen-FineTuned-MBPP on HumanEval:   0%|          | 0/30 [00:00<?, ?it/s]

Fine-tuned Qwen summaries:
{
  "model": "Qwen-FineTuned-MBPP",
  "dataset": "MBPP-Test",
  "num_examples": 30,
  "pass@1": 0.2,
  "pass@3": 0.36666666666666664,
  "execution_accuracy": 0.26666666666666666,
  "avg_codebleu": null
}
{
  "model": "Qwen-FineTuned-MBPP",
  "dataset": "HumanEval",
  "num_examples": 30,
  "pass@1": 0.23333333333333334,
  "pass@3": 0.4666666666666667,
  "execution_accuracy": 0.23333333333333328,
  "avg_codebleu": null
}


In [79]:
# ============================================================
# CODE BLOCK 19B: Inspect Fine-Tuned Qwen Generations
# ============================================================

print("Inspecting Fine-Tuned Qwen MBPP generations:")
inspect_generation_results(
    qwen_ft_mbpp_results,
    n=3
)

print("\nInspecting Fine-Tuned Qwen HumanEval generations:")
inspect_generation_results(
    qwen_ft_humaneval_results,
    n=3
)

Inspecting Fine-Tuned Qwen MBPP generations:
DATASET: MBPP-Test
TASK ID: 485

INSTRUCTION:
Write a function to find the largest palindromic number in the given array.

REFERENCE CODE:
def is_palindrome(n) : 
	divisor = 1
	while (n / divisor >= 10) : 
		divisor *= 10
	while (n != 0) : 
		leading = n // divisor 
		trailing = n % 10
		if (leading != trailing) : 
			return False
		n = (n % divisor) // 10
		divisor = divisor // 100
	return True
def largest_palindrome(A, n) : 
	A.sort() 
	for i in range(n - 1, -1, -1) : 
		if (is_palindrome(A[i])) : 
			return A[i] 
	return -1
--------------------------------------------------------------------------------
GENERATION 1:
def largest_palindrome(arr,n):
    arr.sort()
    for i in range(n-1):
        if (arr[i] == arr[n-i-1]) and (arr[i] > 0):
            return arr[i]
    return -1
Correct: False
Error: Traceback (most recent call last):
  File "/tmp/tmp__pmo263.py", line 8, in <module>
    assert largest_palindrome([1, 232, 54545, 999991], 4)

In [82]:
# ============================================================
# CODE BLOCK 20: Final comparison tables and saved artifacts
# ============================================================

# ------------------------------------------------------------
# Safety fallbacks for optional metadata
# ------------------------------------------------------------

if "token_length_stats" not in globals():
    token_length_stats = {
        "note": "Token length statistics were not collected in this run."
    }

if "tokenization_stats" not in globals():
    tokenization_stats = {
        "note": "Tokenization statistics unavailable."
    }

if "all_summaries" not in globals():
    raise RuntimeError("all_summaries not found. Run baseline and fine-tuned evaluations first.")

if "all_results" not in globals():
    raise RuntimeError("all_results not found. Run baseline and fine-tuned evaluations first.")


# ------------------------------------------------------------
# Build summary dataframe
# ------------------------------------------------------------

summary_df = pd.DataFrame(all_summaries)

summary_csv = os.path.join(
    RESULTS_DIR,
    "stage2_model_comparison_summary.csv"
)

summary_df.to_csv(summary_csv, index=False)

display(summary_df)


# ------------------------------------------------------------
# Flatten detailed result rows
# Handles both:
#   all_results = [row, row, ...]
# and:
#   all_results = [[row, row], [row, row], ...]
# ------------------------------------------------------------

def flatten_result_rows(results):
    flat = []

    for item in results:
        if isinstance(item, dict):
            flat.append(item)
        elif isinstance(item, list):
            flat.extend(flatten_result_rows(item))

    return flat


all_result_rows = flatten_result_rows(all_results)


# ------------------------------------------------------------
# Save detailed generation/evaluation results as JSON
# ------------------------------------------------------------

detailed_json = os.path.join(
    RESULTS_DIR,
    "stage2_detailed_results.json"
)

with open(detailed_json, "w") as f:
    json.dump(all_result_rows, f, indent=2)


# ------------------------------------------------------------
# Save flattened detailed CSV for easier inspection
# ------------------------------------------------------------

flat_rows = []

for row in all_result_rows:
    generations = row.get("generations", [])
    correctness = row.get("correctness", row.get("correct_list", []))
    execution_errors = row.get("execution_errors", [])

    for i, gen in enumerate(generations):
        flat_rows.append({
            "model": row.get("model", ""),
            "dataset": row.get("dataset", ""),
            "task_id": row.get("task_id", ""),
            "source_dataset": row.get("source_dataset", ""),
            "entry_point": row.get("entry_point", ""),
            "generation_id": i + 1,
            "generated_code": gen,
            "correct": correctness[i] if i < len(correctness) else None,
            "execution_error": execution_errors[i] if i < len(execution_errors) else "",
            "instruction": row.get("instruction", ""),
            "reference_code": row.get("reference_code", "")
        })

detailed_csv = os.path.join(
    RESULTS_DIR,
    "stage2_detailed_generations_flat.csv"
)

pd.DataFrame(flat_rows).to_csv(detailed_csv, index=False)


# ------------------------------------------------------------
# Identify best model per dataset by pass@1
# ------------------------------------------------------------

best_by_dataset = {}

if not summary_df.empty and "dataset" in summary_df.columns and "pass@1" in summary_df.columns:
    for dataset_name in summary_df["dataset"].unique():
        dataset_df = summary_df[summary_df["dataset"] == dataset_name].copy()

        if len(dataset_df) > 0:
            best_row = dataset_df.sort_values(
                by="pass@1",
                ascending=False
            ).iloc[0].to_dict()

            best_by_dataset[dataset_name] = best_row


# ------------------------------------------------------------
# Experiment card
# ------------------------------------------------------------

experiment_card = {
    "stage": "Stage 2",
    "task": "Natural Language to Python Code Generation",
    "training_datasets": [
        "MBPP train split only"
    ],
    "validation_datasets": [
        "MBPP validation split only"
    ],
    "evaluation_datasets": [
        "MBPP test split",
        "HumanEval",
        "EvalPlus MBPP+ if loaded and non-empty"
    ],
    "dataset_decision": (
        "The final implementation uses MBPP only for fine-tuning because MBPP is clean, "
        "Python-focused, small enough for Colab, and includes executable test cases. "
        "CodeAlpaca and APPS were investigated but removed from training because they "
        "introduced noise, schema complexity, and implementation risk. HumanEval is retained "
        "only for evaluation to test out-of-domain generalization and avoid benchmark leakage."
    ),
    "human_eval_evaluation_fix": (
        "HumanEval required special handling because it is a function-completion benchmark, "
        "not a pure natural-language-to-code benchmark like MBPP. The final evaluation pipeline "
        "preserves the benchmark entry_point, uses function-name-aware extraction, and executes "
        "the HumanEval prompt context together with the generated completion where required."
    ),
    "fine_tuning_finding": (
        "Safe LoRA fine-tuning was completed successfully, but the pretrained Qwen model remained "
        "stronger overall. The fine-tuned model improved over the initial overfitted fine-tuning "
        "attempt, especially on HumanEval pass@3, but did not surpass the pretrained baseline. "
        "This was analyzed as limited-data overfitting/catastrophic forgetting."
    ),
    "models_compared": [
        "Qwen pretrained",
        "Qwen LoRA fine-tuned on MBPP",
        "DeepSeek pretrained optional if enabled"
    ],
    "base_model_for_finetuning": QWEN_MODEL_NAME,
    "comparison_baseline": DEEPSEEK_MODEL_NAME,
    "metrics": [
        "Pass@1",
        "Pass@3",
        "Execution Accuracy"
    ],
    "pass_k_values": PASS_K_VALUES,
    "demo_mode": DEMO_MODE,
    "num_generations": NUM_GENERATIONS,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "max_total_length": MAX_TOTAL_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "token_length_stats": token_length_stats,
    "tokenization_stats": tokenization_stats,
    "summaries": all_summaries,
    "best_by_dataset": best_by_dataset,
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

experiment_card_path = os.path.join(
    RESULTS_DIR,
    "stage2_experiment_card.json"
)

with open(experiment_card_path, "w") as f:
    json.dump(experiment_card, f, indent=2)


# ------------------------------------------------------------
# Markdown summary report
# ------------------------------------------------------------

markdown_report_path = os.path.join(
    RESULTS_DIR,
    "stage2_results_summary.md"
)

with open(markdown_report_path, "w") as f:
    f.write("# RepoCoder Studio – Stage 2 Results Summary\n\n")

    f.write("## Task\n\n")
    f.write("Natural Language to Python Code Generation\n\n")

    f.write("## Dataset Strategy\n\n")
    f.write("- Training: MBPP train split only\n")
    f.write("- Validation: MBPP validation split only\n")
    f.write("- Evaluation: MBPP test split and HumanEval\n")
    f.write("- Removed: CodeAlpaca and APPS due to noise, schema complexity, and Colab reliability concerns\n\n")

    f.write("## Evaluation Metrics\n\n")
    f.write("- Pass@1\n")
    f.write("- Pass@3\n")
    f.write("- Execution Accuracy\n\n")

    f.write("## Final Comparison\n\n")
    f.write(summary_df.to_markdown(index=False))
    f.write("\n\n")

    f.write("## Best Model by Dataset\n\n")

    if best_by_dataset:
        for dataset_name, row in best_by_dataset.items():
            f.write(
                f"- **{dataset_name}:** {row.get('model')} "
                f"with pass@1 = {row.get('pass@1')}\n"
            )
    else:
        f.write("Best-model summary unavailable.\n")

    f.write("\n## Key Finding\n\n")
    f.write(
        "The pretrained Qwen2.5-Coder model remained the strongest overall model. "
        "Safe LoRA fine-tuning on MBPP completed successfully and improved over the initial "
        "overfitted fine-tuning attempt, but it did not surpass the pretrained baseline. "
        "This was analyzed as limited-data overfitting/catastrophic forgetting.\n\n"
    )

    f.write("## HumanEval Evaluation Fix\n\n")
    f.write(
        "HumanEval initially produced unreliable results because it requires exact function "
        "entry-point preservation and completion-style execution. The final pipeline preserves "
        "entry_point, performs function-aware extraction, and includes prompt context during "
        "HumanEval execution when required.\n\n"
    )

    f.write("## Artifacts Saved\n\n")
    f.write(f"- Summary CSV: `{summary_csv}`\n")
    f.write(f"- Detailed JSON: `{detailed_json}`\n")
    f.write(f"- Detailed flat CSV: `{detailed_csv}`\n")
    f.write(f"- Experiment card: `{experiment_card_path}`\n")


# ------------------------------------------------------------
# Print saved paths
# ------------------------------------------------------------

print("Saved Stage 2 artifacts:")
print("Summary CSV:", summary_csv)
print("Detailed JSON:", detailed_json)
print("Detailed flat CSV:", detailed_csv)
print("Experiment card:", experiment_card_path)
print("Markdown summary:", markdown_report_path)

,model,dataset,num_examples,pass@1,pass@3,execution_accuracy,avg_codebleu
0,Qwen-Pretrained,MBPP-Test,30,0.533333,0.600000,0.488889,None
1,Qwen-Pretrained,HumanEval,30,0.366667,0.400000,0.355556,None
2,Qwen-FineTuned-MBPP,MBPP-Test,30,0.200000,0.366667,0.266667,None
3,Qwen-FineTuned-MBPP,HumanEval,30,0.233333,0.466667,0.233333,None


Saved Stage 2 artifacts:
Summary CSV: /content/RepoCoderStudio_Stage2_MBPP_SafeFT/evaluation_results/stage2_model_comparison_summary.csv
Detailed JSON: /content/RepoCoderStudio_Stage2_MBPP_SafeFT/evaluation_results/stage2_detailed_results.json
Detailed flat CSV: /content/RepoCoderStudio_Stage2_MBPP_SafeFT/evaluation_results/stage2_detailed_generations_flat.csv
Experiment card: /content/RepoCoderStudio_Stage2_MBPP_SafeFT/evaluation_results/stage2_experiment_card.json
Markdown summary: /content/RepoCoderStudio_Stage2_MBPP_SafeFT/evaluation_results/stage2_results_summary.md


In [ ]:
# ============================================================
# CODE BLOCK 21: Gradio demo for Stage 2
# ============================================================

import gradio as gr

def generate_python_code_ui(requirement):
    prompt = build_prompt(requirement)

    solutions = generate_k_solutions(
        qwen_ft_model,
        qwen_ft_tokenizer,
        prompt,
        k=1,
        temperature=0.2,
        top_p=0.9
    )

    if len(solutions) == 0 or solutions[0].strip() == "":
        return "EMPTY_OUTPUT"

    return solutions[0]


example_requirement = "Write a Python function that returns the factorial of a non-negative integer."

demo = gr.Interface(
    fn=generate_python_code_ui,
    inputs=gr.Textbox(
        label="Natural Language Requirement",
        lines=5,
        value=example_requirement
    ),
    outputs=gr.Code(
        label="Generated Python Code",
        language="python"
    ),
    title="RepoCoder Studio - Stage 2 NL to Python Generator",
    description=(
        "Enter a natural-language programming requirement. "
        "The fine-tuned Qwen model generates executable Python code."
    )
)

demo.launch(share=True, debug=True)